In [1]:
from pathlib import Path
from glob import glob
import random
import zipfile
import tqdm
import pandas as pd
import os
zip_path = "Data.zip"      # путь к архиву
extract_dir = "./"  # куда распаковать

# создаём папку, если нет
os.makedirs(extract_dir, exist_ok=True)

# открываем и извлекаем
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"✅ Архив {zip_path} успешно распакован в {extract_dir}")


✅ Архив Data.zip успешно распакован в ./


In [3]:
from urllib.request import urlopen
from PIL import Image
import timm

img = Image.open(urlopen(
    'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/beignets-task-guide.png'
))

model_tm = timm.create_model(
    'eva02_large_patch14_448.mim_m38m_ft_in22k_in1k',
    pretrained=True,
    num_classes=0,  # remove classifier nn.Linear
)
model_tm = model_tm.to("cuda")
model_tm = model_tm.eval()

# get model specific transforms (normalization, resize)
data_config = timm.data.resolve_model_data_config(model_tm)


In [4]:
import torchvision.transforms as transforms2

# Define the required input size
required_size = (448, 448) # (height, width)

transformation = transforms2.Compose([
    transforms2.Resize(required_size), # Resizes the smaller edge to the given size and maintains aspect ratio
    transforms2.CenterCrop(required_size), # Crops the image to the given size from the center

    transforms2.ToTensor(),  # Converts PIL Image to torch.FloatTensor and scales to [0, 1]
    transforms2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def readimgandgetarray(path):

   # try:
        image = Image.open(path).convert('RGB') 

        img_tensor = transformation(image).float()
        img_tensor = img_tensor.unsqueeze_(0)
 

        img_tensor = img_tensor.cuda()
 

        output = model_tm(img_tensor) # output is (batch_size, num_features) shaped tensor
    
        
        return output.cpu().detach().numpy() 
    # except:
    #      return np.zeros(1024)

In [5]:
import pandas as pd

pd.set_option('display.max_rows', 1000)      # показывать все строки
pd.set_option('display.max_columns', 1000)   # показывать все столбцы
pd.set_option('display.max_colwidth', None)
train_data = pd.read_csv("train.csv", sep = ';')
test_data = pd.read_csv("Test (7).csv", sep = ';')


In [6]:
import re
from nltk.corpus import stopwords
from pymorphy3 import MorphAnalyzer
from tqdm.notebook import tqdm
import spacy
import numpy as np

# --- Инициализируем инструменты один раз, чтобы не делать это для каждого текста ---
# Это важно для производительности!
nlp = spacy.load("ru_core_news_sm")
russian_stopwords = set(stopwords.words('russian'))

def preprocess_text(text):
    """
    Функция для лемматизации, удаления стоп-слов и очистки текста с использованием spaCy.
    """
    if not isinstance(text, str) or len(text) <= 6:
        return np.nan

    # Process text using spaCy
    doc = nlp(text.lower()) # Process lowercase text

    clean_tokens = []
    for token in doc:
        # Check if token is alphabetic and not a stop word
        if token.is_alpha and token.text not in russian_stopwords:
            clean_tokens.append(token.lemma_)

    return " ".join(clean_tokens)


train_data['description'] = train_data['description'].apply(preprocess_text)

test_data['description'] = test_data['description'].apply(preprocess_text)


In [7]:
train_data['description'] = train_data['description'].fillna("None")
test_data['description'] = test_data['description'].fillna("None")

In [8]:
train_data.shape[0]

6401

In [9]:
arr = np.empty((train_data.shape[0],1024) )

In [10]:
i = 0
for x in tqdm(train_data['id']):
    z =  readimgandgetarray ('Train/' + str(x)).flatten()
    arr[i] = z
    i += 1



  0%|          | 0/6401 [00:00<?, ?it/s]

In [11]:
from sklearn.decomposition import TruncatedSVD


# Instantiate TruncatedSVD with a desired number of components
svd = TruncatedSVD(n_components=50, random_state=42)

# Fit the SVD model and transform the data
X_transformed = svd.fit_transform(arr)

# Access attributes of the fitted model
print(f"Original data shape: {arr.shape}")
print(f"Transformed data shape: {X_transformed.shape}")
print(f"Explained variance ratio: {svd.explained_variance_ratio_}")
print(f"Singular values: {svd.singular_values_}")
print(f"Components (V^T matrix): {svd.components_.shape}")

Original data shape: (6401, 1024)
Transformed data shape: (6401, 50)
Explained variance ratio: [0.05849555 0.02004322 0.02349475 0.01930539 0.01413704 0.01175333
 0.01091682 0.00946292 0.00944766 0.00878287 0.00823505 0.00778731
 0.00762182 0.00738792 0.0071925  0.00691294 0.00650271 0.00629356
 0.00601199 0.00587856 0.00559115 0.00549466 0.00540512 0.00528526
 0.00509152 0.00504757 0.00470893 0.00458193 0.00455623 0.00445202
 0.00440929 0.00430422 0.00425014 0.00415422 0.00396746 0.00391172
 0.00383055 0.00384025 0.00371918 0.00364398 0.00359444 0.00352198
 0.00343457 0.0033541  0.00330145 0.00328582 0.00322575 0.00320732
 0.00312291 0.00305505]
Singular values: [1323.61873954  701.9997028   633.60494022  560.58741005  480.59398091
  438.45189025  423.26563048  395.44808928  392.21376944  378.37607244
  366.2875161   356.47651492  352.19952304  346.83689722  342.11225489
  335.63414474  326.16764603  320.05565451  314.04114172  309.31565785
  302.000271    298.99400984  296.57206186  

In [12]:
df1 = pd.DataFrame(X_transformed)
combined_df = pd.concat([train_data, df1], axis=1)
combined_df.head()

,id,description,label,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49
0,814469951099,устать жить хотеть полезный помнить гн,Философия и религия,-2.971130,-5.889262,-6.971265,0.201553,9.131372,-1.502359,0.168992,5.395300,-4.778130,3.455151,-1.815785,-6.568954,-15.170149,-16.804710,7.789484,7.537541,6.764469,4.524818,4.755453,-4.239569,-2.203099,-0.277907,-9.377923,4.736896,-5.717084,-0.261206,-4.725472,2.499590,-1.491958,0.454736,4.692570,4.523035,0.796685,3.674493,4.087040,0.924897,0.846592,-3.028437,-1.786508,2.247336,-4.055994,0.381974,1.000115,2.403050,1.578587,-0.302447,-3.126831,-1.526206,0.463572,-0.634765
1,849433210092,None,Торговля и объявления,-20.569196,-0.551441,-6.053828,-6.832979,4.671089,1.864230,1.699865,-5.468234,3.204165,-3.096345,8.056549,-12.722799,5.105375,5.495707,-0.755472,-9.929283,6.436697,7.045133,-8.155952,2.936009,-6.101162,-5.679013,0.963965,2.255545,-0.825669,1.686000,-8.386044,3.431922,4.906952,-0.723731,-0.265923,0.729077,2.000554,0.569975,3.197168,0.542042,-0.705706,5.143203,0.992787,-0.894084,1.653073,-2.214700,7.235931,3.562414,-1.780189,0.394398,-1.497801,-4.855083,1.531281,-4.971940
2,852458632411,мир ваш дом дорогой друг хранить господь,Философия и религия,-27.218104,-9.164081,3.337289,-4.140871,11.373103,-0.410532,-4.929092,4.786883,-0.108527,-0.582016,1.601082,0.624694,-7.894957,1.369759,3.718079,0.834584,-4.674021,-1.809273,-3.661189,3.097574,3.150692,-1.910058,3.956962,-2.147839,4.297485,5.087630,-2.467553,-4.328582,5.134421,-0.948379,0.046057,1.530867,-4.109867,4.981377,-0.116146,2.352852,-3.448644,-1.679051,1.547979,-2.954075,-1.069785,-5.588120,3.706944,1.887479,-5.418427,-5.269034,1.925852,-0.698188,0.119606,-2.941960
3,860243294215,альбом,Животные,-0.268429,-2.565711,-3.795832,1.265914,2.368235,0.140004,-1.107816,-1.627469,-0.909221,-0.635025,0.636075,-1.547772,-3.046122,0.272422,-0.946814,-0.050713,3.388656,1.273798,0.792260,-2.919814,1.248896,-0.935383,1.298207,1.613109,1.025989,2.517481,2.998915,0.966137,4.148508,-2.861855,-2.057636,-1.637658,0.844422,0.292553,-0.942817,2.527392,-1.180861,-0.491877,3.657146,0.830273,-0.366155,1.676355,0.496276,-0.876644,0.253975,3.600264,1.458750,-0.864860,0.557473,-2.458361
4,861555576675,умный некоторый человек,Животные,-4.662194,-5.513715,-2.727942,1.423381,4.749343,3.358798,3.846032,-4.264952,-4.138183,3.540212,4.540810,-6.001133,-0.350830,0.513978,-0.072012,-2.073393,3.179624,-3.281036,-3.508634,-0.928460,-3.136996,-1.450336,3.630643,-0.262487,2.535564,-3.844187,3.397653,3.866672,-1.059438,5.478908,0.324570,-3.743469,2.389936,-1.400020,4.981021,0.526107,-2.371134,2.319089,0.878249,2.097977,-0.538983,-0.274159,-2.052757,2.530628,-3.071983,-0.762713,4.310670,-2.567128,1.145322,-4.357711


In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
combined_df['label']  = le.fit_transform(combined_df['label'])

In [15]:
combined_df['description'] = combined_df['description'].fillna('MISSING_VALUE').astype(str)
X=combined_df.drop([ 'label'],axis=1)
y=combined_df['label']
X.head()

,id,description,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49
0,814469951099,устать жить хотеть полезный помнить гн,-2.971130,-5.889262,-6.971265,0.201553,9.131372,-1.502359,0.168992,5.395300,-4.778130,3.455151,-1.815785,-6.568954,-15.170149,-16.804710,7.789484,7.537541,6.764469,4.524818,4.755453,-4.239569,-2.203099,-0.277907,-9.377923,4.736896,-5.717084,-0.261206,-4.725472,2.499590,-1.491958,0.454736,4.692570,4.523035,0.796685,3.674493,4.087040,0.924897,0.846592,-3.028437,-1.786508,2.247336,-4.055994,0.381974,1.000115,2.403050,1.578587,-0.302447,-3.126831,-1.526206,0.463572,-0.634765
1,849433210092,None,-20.569196,-0.551441,-6.053828,-6.832979,4.671089,1.864230,1.699865,-5.468234,3.204165,-3.096345,8.056549,-12.722799,5.105375,5.495707,-0.755472,-9.929283,6.436697,7.045133,-8.155952,2.936009,-6.101162,-5.679013,0.963965,2.255545,-0.825669,1.686000,-8.386044,3.431922,4.906952,-0.723731,-0.265923,0.729077,2.000554,0.569975,3.197168,0.542042,-0.705706,5.143203,0.992787,-0.894084,1.653073,-2.214700,7.235931,3.562414,-1.780189,0.394398,-1.497801,-4.855083,1.531281,-4.971940
2,852458632411,мир ваш дом дорогой друг хранить господь,-27.218104,-9.164081,3.337289,-4.140871,11.373103,-0.410532,-4.929092,4.786883,-0.108527,-0.582016,1.601082,0.624694,-7.894957,1.369759,3.718079,0.834584,-4.674021,-1.809273,-3.661189,3.097574,3.150692,-1.910058,3.956962,-2.147839,4.297485,5.087630,-2.467553,-4.328582,5.134421,-0.948379,0.046057,1.530867,-4.109867,4.981377,-0.116146,2.352852,-3.448644,-1.679051,1.547979,-2.954075,-1.069785,-5.588120,3.706944,1.887479,-5.418427,-5.269034,1.925852,-0.698188,0.119606,-2.941960
3,860243294215,альбом,-0.268429,-2.565711,-3.795832,1.265914,2.368235,0.140004,-1.107816,-1.627469,-0.909221,-0.635025,0.636075,-1.547772,-3.046122,0.272422,-0.946814,-0.050713,3.388656,1.273798,0.792260,-2.919814,1.248896,-0.935383,1.298207,1.613109,1.025989,2.517481,2.998915,0.966137,4.148508,-2.861855,-2.057636,-1.637658,0.844422,0.292553,-0.942817,2.527392,-1.180861,-0.491877,3.657146,0.830273,-0.366155,1.676355,0.496276,-0.876644,0.253975,3.600264,1.458750,-0.864860,0.557473,-2.458361
4,861555576675,умный некоторый человек,-4.662194,-5.513715,-2.727942,1.423381,4.749343,3.358798,3.846032,-4.264952,-4.138183,3.540212,4.540810,-6.001133,-0.350830,0.513978,-0.072012,-2.073393,3.179624,-3.281036,-3.508634,-0.928460,-3.136996,-1.450336,3.630643,-0.262487,2.535564,-3.844187,3.397653,3.866672,-1.059438,5.478908,0.324570,-3.743469,2.389936,-1.400020,4.981021,0.526107,-2.371134,2.319089,0.878249,2.097977,-0.538983,-0.274159,-2.052757,2.530628,-3.071983,-0.762713,4.310670,-2.567128,1.145322,-4.357711


In [17]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# ... (assuming train_data is defined and the DataFrame creation code is present) ...
# Example of assumed data setup:
# X = pd.DataFrame(train_data['description'], columns=['description'])
# y = train_data['label']
text_features = ['description'] 


X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# --- Code using CatBoost Pool ---

# 1. Create a Pool object for the training data
# We explicitly pass the text_features list to the Pool constructor
train_pool = Pool(
    data=X_train, 
    label=y_train, 
    text_features=text_features
)

# 2. Create a Pool object for the validation data
# The text_features must be defined identically to the training pool
eval_pool = Pool(
    data=X_val, 
    label=y_val,
    text_features=text_features
)

# 3. Initialize the model (parameters remain the same)
model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.03,
    depth=7,
    eval_metric='Accuracy',
    custom_metric=['F1'], 
 task_type="GPU",
    random_seed=42,
    verbose=10
)

# 4. Fit the model using the Pool objects
# Instead of passing X_train, y_train, X_val, y_val, we pass the Pool objects
model.fit(
    X=train_pool,          # Pass the training Pool
    eval_set=eval_pool,    # Pass the validation Pool
    early_stopping_rounds=150
    # Note: text_features argument is no longer needed in .fit() 
    # because it was already defined when creating the Pool objects.
)



0:	learn: 0.4943359	test: 0.4793130	best: 0.4793130 (0)	total: 63.3ms	remaining: 2m 6s
10:	learn: 0.5289063	test: 0.5136612	best: 0.5144418 (7)	total: 375ms	remaining: 1m 7s
20:	learn: 0.5507813	test: 0.5339578	best: 0.5355191 (18)	total: 680ms	remaining: 1m 4s
30:	learn: 0.5755859	test: 0.5550351	best: 0.5550351 (30)	total: 994ms	remaining: 1m 3s
40:	learn: 0.5986328	test: 0.5815769	best: 0.5815769 (40)	total: 1.31s	remaining: 1m 2s
50:	learn: 0.6187500	test: 0.6010929	best: 0.6010929 (50)	total: 1.63s	remaining: 1m 2s
60:	learn: 0.6339844	test: 0.6128025	best: 0.6128025 (60)	total: 1.94s	remaining: 1m 1s
70:	learn: 0.6462891	test: 0.6213895	best: 0.6213895 (70)	total: 2.25s	remaining: 1m 1s
80:	learn: 0.6556641	test: 0.6354411	best: 0.6354411 (80)	total: 2.56s	remaining: 1m
90:	learn: 0.6652344	test: 0.6432475	best: 0.6432475 (90)	total: 2.88s	remaining: 1m
100:	learn: 0.6744141	test: 0.6471507	best: 0.6479313 (99)	total: 3.2s	remaining: 1m
110:	learn: 0.6828125	test: 0.6471507	best:

In [18]:
from sklearn.metrics import f1_score

y_pred = model.predict(X_val)
acc=f1_score(y_val, y_pred, average='macro')
print(f"F1: {acc:.4f}")


F1: 0.4510


In [19]:
import pandas as pd

importances = model.get_feature_importance()
feature_names = X_train.columns

fi = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

fi.head(len(fi))

,feature,importance
1,description,14.136973
2,0,9.080590
6,4,8.170340
9,7,7.555976
5,3,6.849564
4,2,4.458694
7,5,4.256235
8,6,2.954431
13,11,2.830423
10,8,2.767663


In [20]:
arr = np.empty((test_data.shape[0],1024) )

In [21]:
i = 0
for x in tqdm(test_data['id']):
    z =  readimgandgetarray ('Test/' + str(x)).flatten()
    arr[i] = z
    i += 1

test_transformed =svd.transform(arr)


df1 = pd.DataFrame(test_transformed)
combined_test  = pd.concat([test_data, df1], axis=1)


  0%|          | 0/2565 [00:00<?, ?it/s]

In [22]:
combined_test['description'] = combined_test['description'].fillna('MISSING_VALUE').astype(str)


In [23]:
#X_test = pd.DataFrame(test_data['description'], columns=['description'])

test_pool = Pool(
    data=combined_test, 
    text_features=text_features # text_features is ['description']
)

y_pred = model.predict(test_pool)


In [24]:
test_data['description']

0                                                   None
1       танец живот невеста лишить дара речь жених смотр
2                                          дмитpий лeвин
3                       продам ваз карбюратор газ бензин
4                  память стать резко ухудшаться пропить
                              ...                       
2560                зимний утро автор татьяна прохоренко
2561      петровский село городской поселение пушкиногор
2562         храм честь воскресение христова посвящённый
2563       хороший подарок любимый новый год ласковый иг
2564                              ручной роспись потолок
Name: description, Length: 2565, dtype: object

In [26]:
y_pred

array([[15],
       [15],
       [14],
       ...,
       [14],
       [15],
       [19]], shape=(2565, 1))

In [25]:
subm = pd.read_csv("submission (3).csv", sep = ';')
subm['label'] = le.inverse_transform(y_pred)
subm.to_csv("submission.csv", index = None, sep = ';')

/opt/jupyter/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:151: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [27]:
subm

,id,label
0,909340245742,Развлечения и юмор
1,909342962411,Развлечения и юмор
2,909343087161,Путешествия
3,909344193109,Торговля и объявления
4,909346841420,Кулинария
...,...,...
2560,970656513024,Путешествия
2561,970691145216,Путешествия
2562,970699981568,Путешествия
2563,970757032704,Развлечения и юмор
